# Registro de experimentos del proyecto de Ecobici

Cada modelo de la Semana 10 (persistencia, Ridge, gradient boosting y red neuronal) queda como una corrida en MLflow, en el experimento `ecobici-rebalanceo`, con la preparación y la división por fecha de [`pipeline_proyecto.py`](../../semana-09-sesion-02/plantilla-pipeline/pipeline_proyecto.py). Al final se marca el modelo final, se carga desde el registro y se comprueba que reproduce su métrica. Exporta `corridas_proyecto.csv`.

El registro vive en el servidor de MLflow (variable de entorno `MLFLOW_TRACKING_URI`, puerto 5050).

In [1]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

import sys

# pipeline_proyecto.py vive en la carpeta del pipeline de la Semana 9, Sesion 2.
sys.path.insert(0, "../../semana-09-sesion-02/plantilla-pipeline")
from pipeline_proyecto import RUTA_DATOS, ULTIMO_DIA_ENTRENAMIENTO, cargar_y_dividir, construir_preparacion

# MLflow avisa que el modelo se guarda sin firma; importa hasta la Semana 12.
logging.getLogger("mlflow").setLevel(logging.ERROR)

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Entrenamiento: (230400, 11) Prueba: (67100, 11)


## La huella de los datos y el experimento

In [2]:
def huella(ruta):
    """Los primeros 12 caracteres del hash MD5 del archivo."""
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella(RUTA_DATOS)
FECHA_CORTE = f"2026-08-{ULTIMO_DIA_ENTRENAMIENTO}"
EXPERIMENTO = "ecobici-rebalanceo"

print("Registro de MLflow:", mlflow.get_tracking_uri())
experimento = mlflow.set_experiment(EXPERIMENTO)
print("Experimento:", experimento.name, "| huella de los datos:", huella_datos, "| entrenamiento hasta:", FECHA_CORTE)

Registro de MLflow: http://127.0.0.1:5050
Experimento: ecobici-rebalanceo | huella de los datos: 968012844003 | entrenamiento hasta: 2026-08-24


## La función que entrena y registra cada corrida

In [3]:
def parametros_division():
    return {
        "division": "por fecha",
        "fecha_corte": FECHA_CORTE,
        "huella_datos": huella_datos,
        "filas_entrenamiento": len(X_ent),
    }


def evaluar(nombre, modelo):
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    mlflow.start_run(run_name=nombre)
    mlflow.log_params({**modelo.get_params(), **parametros_division()})
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    mlflow.log_metrics({"mae_entrenamiento": mae_ent, "mae_prueba": mae_prueba, "segundos": segundos})
    mlflow.sklearn.log_model(pipeline, "modelo")
    mlflow.end_run()
    print(f"{nombre:22s} entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:7.4f}   {segundos:6.2f} s")

## Línea base: persistencia

Los próximos 30 minutos iguales a los últimos 30 (la franja actual más la anterior). Es una regla, no aprende nada: se registran su definición, la división y sus métricas, pero no hay un modelo entrenado que guardar.

In [4]:
def persistencia(tabla):
    return tabla["salidas"] + tabla["salidas_lag_15m"].fillna(0)


mlflow.start_run(run_name="persistencia 30 min")
mlflow.log_params({"regla": "salidas + salidas_lag_15m", **parametros_division()})
mae_ent = mean_absolute_error(y_ent, persistencia(X_ent))
mae_prueba = mean_absolute_error(y_prueba, persistencia(X_prueba))
mlflow.log_metrics({"mae_entrenamiento": mae_ent, "mae_prueba": mae_prueba, "segundos": 0.0})
mlflow.end_run()
print(f"{'persistencia 30 min':22s} entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:7.4f}")

persistencia 30 min    entrenamiento  1.1527   prueba  1.2278


## Los modelos de la Semana 10

In [5]:
evaluar("ridge", Ridge(alpha=1.0))
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42))
evaluar("red neuronal", MLPRegressor(hidden_layer_sizes=(64,), max_iter=200, early_stopping=True, random_state=42))

ridge                  entrenamiento  1.0117   prueba  1.0632     0.31 s


gradient boosting      entrenamiento  0.9739   prueba  1.0278    18.69 s


red neuronal           entrenamiento  0.9649   prueba  1.0209    88.74 s


## El registro, ordenado

In [6]:
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.mae_prueba ASC"])
columnas = ["run_id", "tags.mlflow.runName", "params.fecha_corte", "params.huella_datos",
            "metrics.mae_entrenamiento", "metrics.mae_prueba", "metrics.segundos"]
print(corridas[columnas].to_string(index=False))

                          run_id tags.mlflow.runName params.fecha_corte params.huella_datos  metrics.mae_entrenamiento  metrics.mae_prueba  metrics.segundos
ee90d897703440af8105d725b4ab29b5        red neuronal         2026-08-24        968012844003                   0.964900            1.020914         88.741673
b3cdcd9d46894965b2f2b9fbc5fde396   gradient boosting         2026-08-24        968012844003                   0.973900            1.027782         18.685767
9f19806e7aa4445a8440532ad898dd78               ridge         2026-08-24        968012844003                   1.011668            1.063220          0.311701
025fe730433645b9aa1772fa5c7f7118 persistencia 30 min         2026-08-24        968012844003                   1.152665            1.227809          0.000000


## El modelo final y su reproducción

El final es el gradient boosting, como en la decisión de la Semana 10: la red neuronal le gana por 0.007 en prueba, pero tarda más del doble en entrenar y no permite leer qué columnas mandan.

In [7]:
FINAL = "gradient boosting"
run_final = corridas.loc[corridas["tags.mlflow.runName"] == FINAL, "run_id"].iloc[0]
mlflow.MlflowClient().set_tag(run_final, "candidato", "final")

modelo_final = mlflow.sklearn.load_model("runs:/" + run_final + "/modelo")
mae_reproducido = mean_absolute_error(y_prueba, modelo_final.predict(X_prueba))
mae_registrado = corridas.loc[corridas["run_id"] == run_final, "metrics.mae_prueba"].iloc[0]
print("run_id final:", run_final)
print(f"MAE registrado {mae_registrado:.6f}   reproducido {mae_reproducido:.6f}   coincide: {abs(mae_reproducido - mae_registrado) < 1e-9}")

run_id final: b3cdcd9d46894965b2f2b9fbc5fde396
MAE registrado 1.027782   reproducido 1.027782   coincide: True


## La tabla de corridas

In [8]:
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.mae_prueba ASC"])
corridas.to_csv("corridas_proyecto.csv", index=False)
print("corridas_proyecto.csv:", corridas.shape)

corridas_proyecto.csv: (4, 59)
